# Hologuess examples by problem

Pick an example that matches your data. Each section gives the input, the public Hologuess call, and a Sage result you can inspect or reuse. See the [quickstart](quickstart.ipynb) for a first run. Its [LaTeX notation section](quickstart.ipynb#Change-or-copy-the-LaTeX-notation) shows how to set symbols and copy LaTeX.

Use a **SageMath** kernel with Hologuess and `ore_algebra` installed. The first cell loads the imports used below. All examples here run offline with those packages.

[Recurrences](#Recurrences-with-polynomial-coefficients) · [Generating functions](#Generating-functions) · [Taylor series](#Differential-equation-from-Taylor-coefficients) · [Other coefficient domains](#Other-coefficient-domains) · [Two-variable data](#Two-variable-data) · [q-recurrences](#q-recurrence-from-Laurent-polynomials) · [Custom actions](#Custom-actions) · [Benchmarks](#Benchmarks)

## Recurrences with polynomial coefficients

Find a recurrence for $a_n=1/(n+1)$. Here the forward shift $E$ is named `Sn` in Ore algebra and displayed as $S$. `path=` lists the pairs of bounds to try, in order. Each pair is `(max_order, max_coefficient_degree)`. Here `path=[(1, 1)]` asks for a single search that allows $p_0(n)+p_1(n)S$, with both polynomials of degree at most one. If you supply several pairs, the search stops at the first pair that yields an operator. [Ore algebra rational-input doctest](https://github.com/mkauers/ore_algebra/blob/a9bce94221c11ae142a0e95e26baa8987783d246/src/ore_algebra/guessing.py#L7)

The list below gives $a_0,\ldots,a_9$. At order one each window is a pair $(a_n,a_{n+1})$. Setting `holdout=2` reserves the final two windows for an exact check.

- **Guess** using the windows starting at $n=0,\ldots,6$. Together they use $a_0,\ldots,a_7$.
- **Check** the returned recurrence on the windows starting at $n=7,8$. These windows reach the withheld values $a_8$ and $a_9$.

In [1]:
%display latex
from sage.all import QQ, ZZ, GF, LaurentPolynomialRing, PolynomialRing, QuadraticField
from sage.all import binomial, factorial, fibonacci
from ore_algebra import OreAlgebra
from hologuess import ModuleProblem, guess, guess_qsequence, solve_module

shift = OreAlgebra(QQ["n"], "Sn")
values = [1 / QQ(k + 1) for k in range(10)]
rational = guess(values, shift, path=[(1, 1)], holdout=2)
rational.operator

(n + 2)*Sn - n - 1

Read the result as $(n+2)a_{n+1}-(n+1)a_n=0$. For harmonic numbers $H_n=\sum_{j=1}^n1/j$, allow order 2 and the same coefficient degree. [RISC Guess handbook, harmonic-number input cell](https://www3.risc.jku.at/research/combinat/software/ergosum/RISC/Guess/demo.nb)

In [2]:
harmonic = [sum(1 / QQ(j) for j in range(1, k + 1)) for k in range(19)]
guess(harmonic, shift, path=[(2, 1)], holdout=8).operator

(n + 2)*Sn^2 + (-2*n - 3)*Sn + n + 1

### A constant-coefficient recurrence

This is SymPy's order-three test sequence, beginning with $1,1,1$. The first 18 values are used for fitting and five more are reserved. Set the degree to zero for constant coefficients. [SymPy recurrence test](https://github.com/sympy/sympy/blob/16fa855354eb7bcabd3fe10993841e03b1382692/sympy/concrete/tests/test_guess.py#L27-L39)

In [3]:
values = [1, 1, 1]
for _ in range(20):
    values.append(5 * values[-1] - 3 * values[-2] + 8 * values[-3])

cubic = guess(values, shift, path=[(3, 0)], holdout=5)
cubic.operator

Sn^3 - 5*Sn^2 + 3*Sn - 8

The next input from the same test gives an empty operator tuple. With `order=5` and no fixed `path`, Hologuess tries constant-coefficient orders 1 through 5. The status describes this search on the supplied data.

In [4]:
values = [0, 2, 15, 74, 12, 3, 0, 1, 2, 85, 4, 5, 63]
irregular = guess(values, shift, order=5, degree=0, holdout=2)
irregular.status, irregular.operators

('no_relations', ())

## Generating functions

Use `C` as a commutative generator for the unknown series $C(x)$. First infer a quadratic equation from ten Catalan numbers. In this search the equation can contain $C^2$ and its coefficients are polynomials of degree at most one in $x$. [Ore algebra Catalan doctest](https://github.com/mkauers/ore_algebra/blob/a9bce94221c11ae142a0e95e26baa8987783d246/src/ore_algebra/guessing.py#L158-L162)

In [5]:
x_ring = QQ["x"]
series = OreAlgebra(x_ring, "C")
catalan = [binomial(2 * k, k) // (k + 1) for k in range(10)]
guess(catalan, series, path=[(2, 1)], holdout=2).operator

x*C^2 - C + 1

For the shifted Fibonacci numbers $F_5,\ldots,F_{14}$, use an equation linear in $C$. If the result is $p_0(x)+p_1(x)C$, then $C(x)=-p_0(x)/p_1(x)$. [SymPy rational generating-function test](https://github.com/sympy/sympy/blob/16fa855354eb7bcabd3fe10993841e03b1382692/sympy/concrete/tests/test_guess.py#L50-L53)

In [6]:
values = [fibonacci(k) for k in range(5, 15)]
equation = guess(values, series, path=[(1, 2)], holdout=2).operator
constant, linear = equation.list()
-constant / linear

(-3*x - 5)/(x^2 + x - 1)

## Differential equation from Taylor coefficients

The input is $[a_0,a_1,\ldots]$ for $f(x)=\sum a_kx^k$. The derivative $D$ is named `Dx` in Ore algebra. Here we use the ternary hypergeometric coefficient ratio from a Maple gfun test and search for a differential equation of order and coefficient degree at most 3. [Maple gfun differential test](https://gitlab.inria.fr/salvy/gfun/-/blob/f970022b70c55bffe028cefb7232a33964225b90/test/guessing_list_and_series.tst#L145)

In [7]:
taylor = [QQ.one()]
for k in range(39):
    ratio = QQ(27*k**2 + 81*k + 60) / (4*k**2 + 22*k + 30)
    taylor.append(taylor[-1] * ratio)

derivatives = OreAlgebra(x_ring, "Dx")
guess(taylor, derivatives, path=[(3, 3)], holdout=4).operator

(x^3 - 4/27*x^2)*Dx^3 + (7*x^2 - 26/27*x)*Dx^2 + (92/9*x - 10/9)*Dx + 20/9

## Other coefficient domains

### A rational parameter

For $a_n=1/(n+t)+t^n$, construct the shift algebra over $\mathbb{Q}[t][n]$. The order and degree bounds refer to $E$ and $n$, respectively. `parameter_degree` bounds the degree in $t$ of the coefficients after clearing denominators. [Ore algebra rational-parameter doctest](https://github.com/mkauers/ore_algebra/blob/a9bce94221c11ae142a0e95e26baa8987783d246/src/ore_algebra/guessing.py#L153-L157)

In [8]:
parameters = QQ["t"]
t = parameters.gen()
parameter_shift = OreAlgebra(parameters["n"], "Sn")
values = [1 / (k + t) + t**k for k in range(100)]
guess(values, parameter_shift, path=[(2, 2)], parameter_degree=4, holdout=8).operator

((t - 1)*n^2 + (2*t^2 + t - 2)*n + t^3 + 2*t^2)*Sn^2 + ((-t^2 + 1)*n^2 + (-2*t^3 - 3*t^2 + 2*t + 1)*n - t^4 - 3*t^3 - t^2 + t)*Sn + (t^2 - t)*n^2 + (2*t^3 - t)*n + t^4 + t^3 - t^2

### A finite field and an algebraic number field

Reduce the Catalan numbers modulo 1009 and fit a recurrence over $\mathbb{F}_{1009}$. [Maple gfun finite-field test](https://gitlab.inria.fr/salvy/gfun/-/blob/f970022b70c55bffe028cefb7232a33964225b90/test/guessing_architecture_characterization.tst#L39)

In [9]:
F = GF(1009)
finite_shift = OreAlgebra(F["n"], "Sn")
values = [F(binomial(2 * k, k) // (k + 1)) for k in range(32)]
guess(values, finite_shift, path=[(1, 1)], holdout=4).operator

(504*n + 1008)*Sn + 2*n + 1

Fit the geometric sequence $1,\alpha,\alpha^2,\ldots$ in the exact field $\mathbb{Q}(\alpha)$ with $\alpha^2=2$. [Maple gfun algebraic field test](https://gitlab.inria.fr/salvy/gfun/-/blob/f970022b70c55bffe028cefb7232a33964225b90/test/guessing_domain.tst#L31)

In [10]:
K = QuadraticField(2, "alpha")
alpha = K.gen()
quadratic_shift = OreAlgebra(K["n"], "Sn")
values = [alpha**k for k in range(32)]
guess(values, quadratic_shift, path=[(1, 0)], holdout=4).operator

-1/2*alpha*Sn + 1

### Several relations in one box

For $(5n+3)F_n^3/(3n+4)$ over $\mathbb{F}_{1091}$, a single fixed box gives two independent operators. `result.operators` displays the basis as a tuple. The box bounds are part of the answer. [Ore algebra finite-field kernel doctest](https://github.com/mkauers/ore_algebra/blob/a9bce94221c11ae142a0e95e26baa8987783d246/src/ore_algebra/guessing.py#L342-L350)

In [11]:
F = GF(1091)
finite_shift = OreAlgebra(F["n"], "Sn")
values = [F(QQ(5*k + 3) / (3*k + 4) * fibonacci(k)**3) for k in range(200)]
finite = guess(values, finite_shift, path=[(5, 3)], holdout=8)
finite.operators

((154*n^3 + 77*n^2 + 1090*n + 1034)*Sn^5 + (274*n^3 + 227*n^2 + 851*n + 413)*Sn^4 + (141*n^3 + 346*n^2 + 890*n + 226)*Sn^3 + (410*n^3 + 756*n^2 + 726*n + 558)*Sn^2 + (180*n^3 + 360*n^2 + 68*n + 849)*Sn + 736*n^3 + 458*n^2 + 1,
 (113*n^3 + 1005*n^2 + 458*n + 104)*Sn^5 + (599*n^3 + 873*n^2 + 791*n + 514)*Sn^4 + (872*n^3 + 853*n^2 + 918*n + 446)*Sn^3 + (166*n^3 + 416*n^2 + 331*n + 124)*Sn^2 + (745*n^3 + 668*n^2 + 329*n + 938)*Sn + 938*n^3 + 615*n^2 + n)

You can inspect the fitting and validation status separately. Here both basis operators pass the eight reserved windows.

In [12]:
finite.status, finite.metadata["holdout_observations"], finite.metadata["exact_holdout"]

('validated', 8, True)

## Two-variable data

The row index $n$ is the sequence index and the column index $j$ is the power of $y$. For the data $a_{n,j}=\binom{j}{n}$, `Sn` shifts $n$ and `Dy` differentiates the series in $y$. They appear as $S$ and $D$ in the LaTeX output. Set a bound for each generator with the pair `(1, 1)`, followed by the bound on total coefficient degree. [Ore algebra mixed shift and derivative doctest](https://github.com/mkauers/ore_algebra/blob/a9bce94221c11ae142a0e95e26baa8987783d246/src/ore_algebra/guessing.py#L1345)

In [13]:
mixed_ring = PolynomialRing(QQ, ("n", "y"))
mixed = OreAlgebra(mixed_ring, "Sn", "Dy")
values = [[binomial(j, n) for j in range(10)] for n in range(10)]
guess(values, mixed, path=[((1, 1), 1)], holdout=5).operators

((y - 1)*Sn + y, (n + 1)*Sn + (-y)*Dy + n, (y - 1)*Sn*Dy + Sn + y*Dy + 1)

To look for a recurrence in just the second index of $f(n,k)=n!\,2^k$, set the first shift's bound to zero. The display indexes two shifts as $S_n$ and $S_k$. This uses the RISC handbook's data and restricts its two-index search to one direction. [RISC Guess handbook, two-index example](https://www3.risc.jku.at/research/combinat/software/ergosum/RISC/Guess/demo.nb)

In [14]:
grid_ring = PolynomialRing(QQ, ("n", "k"))
grid_shifts = OreAlgebra(grid_ring, "Sn", "Sk")
values = [[factorial(n) * 2**k for k in range(5)] for n in range(5)]
guess(values, grid_shifts, path=[((0, 1), 0)], holdout=5).operator

Sk - 2

## q-recurrence from Laurent polynomials

Take $a_n=\prod_{j=1}^n(1+q^j)$ with $a_0=1$. The operators satisfy $Ef(n)=f(n+1)$, $Qf(n)=q^nf(n)$, and $EQ=qQE$. We use 20 Laurent polynomials for fitting and reserve the final eight values ourselves. [RISC qGeneratingFunctions notebook, QREGuess example](https://www3.risc.jku.at/research/combinat/software/ergosum/RISC/qGeneratingFunctions/demo.nb)

In [15]:
laurent = LaurentPolynomialRing(ZZ, "q")
q = laurent.gen()
values = [laurent.one()]
for k in range(1, 28):
    values.append(values[-1] * (1 + q**k))

fit_terms = 20
q_result = guess_qsequence(values[:fit_terms], max_order=1, max_degree=1)
operator = q_result.to_ore()
operator

-E + q*Q + 1

The operator is a scalar multiple of $E-(1+qQ)$. This means $a_{n+1}=(1+q^{n+1})a_n$. The two coefficients of the returned operator are functions of $q$ and $Q$. Evaluate them at $Q=q^n$ to check the eight windows that reach an unseen value.

In [16]:
left, right = operator.list()
[left(q, q**n) * values[n] + right(q, q**n) * values[n + 1]
 for n in range(fit_terms - 1, len(values) - 1)]

[0, 0, 0, 0, 0, 0, 0, 0]

## Custom actions

Use `ModuleProblem` when you can evaluate the terms of a relation directly. For the Somos-4 sequence, look for a linear combination of the three quadratic features $a_na_{n+4}$, $a_{n+1}a_{n+3}$ and $a_{n+2}^2$. Each feature contributes one column. The inputs are exact rationals and the final three rows are reserved. [FriCAS Somos-4 test](https://github.com/fricas/fricas/blob/f6c146ab9517fd88a3ebe49c1e996616aee61c61/src/input/mantepse.input#L172)

In [17]:
somos = [QQ.one()] * 4
for n in range(15):
    somos.append((somos[n + 1] * somos[n + 3] + somos[n + 2]**2) / somos[n])

labels = ("a(n) a(n+4)", "a(n+1) a(n+3)", "a(n+2)^2")
columns = [
    [somos[n] * somos[n + 4] for n in range(15)],
    [somos[n + 1] * somos[n + 3] for n in range(15)],
    [somos[n + 2]**2 for n in range(15)],
]
somos_result = solve_module(ModuleProblem(columns, labels, training_rows=12))
somos_result.valid_basis

((-1, 1, 1),)

The coefficients are in the same order as the columns. The relation is $a_na_{n+4}=a_{n+1}a_{n+3}+a_{n+2}^2$. `basis` gives the fitting kernel, and `valid_basis` gives its intersection with the reserved equations. If those equations select a smaller space, use fresh observations for another independent check.

### Primitive Hodge polynomials

For cubic hypersurfaces, let $P_n(x)$ be the primitive Hodge polynomial in dimension $n$. The degree of a graded Jacobian-ring piece is $3(p+1)-(n+2)$ for the coefficient of $x^p$. The four features are $P_n$, $P_{n+1}$, $d P_{n+1}/dx$, and $P_{n+2}$.

In [18]:
x = x_ring.gen()
z = QQ["z"].gen()
P = []
for n in range(25):
    jacobian = (1 + z)**(n + 2)
    P.append(sum(jacobian[3*(p + 1) - (n + 2)] * x**p
                 for p in range(n + 1) if 3*(p + 1) >= n + 2))
P[:5]

[2, x + 1, 6*x, 5*x^2 + 5*x, x^3 + 20*x^2 + x]

Multiply each feature by the six monomials $n^a x^b$ with $a\leq 1$ and $b\leq 2$. The tuple below gives the resulting coefficient polynomials of the four features in order. The first 12 dimension rows are used for fitting.

In [19]:
features = [P[:-2], P[1:-1], [p.derivative() for p in P[1:-1]], P[2:]]
labels = [(j, a, b) for j in range(4) for a in range(2) for b in range(3)]
columns = [[n**a * x**b * value for n, value in enumerate(features[j])]
           for j, a, b in labels]

hodge = solve_module(ModuleProblem(columns, labels, training_rows=12))
u, v = PolynomialRing(QQ, ("n", "x")).gens()
weights = hodge.valid_basis[0]
tuple(sum(w * u**a * v**b for w, (i, a, b) in zip(weights, labels) if i == j)
      for j in range(4))

(-2*n*x - 6*x, -2*n*x + n - 3*x, 3*x^2 - 3*x, n + 3)

## Benchmarks

The cold races use fresh processes and compare exact normalized operators or complete bounded spaces. Their reports include the original inputs, timing scope, and reproduction commands.

- **Jones.** FLINT Hologuess was faster on both knots where native Ore returned a verified result. The $5_2$ and $6_1$ native Ore entries are $\infty$ because the runner stopped them at the shared 60-second cutoff with no verified result. Hologuess matched the stored operators in both cases. [Jones races](../benchmarks/results/2026-11-26-flint-races/report.md)
- **Differential operators.** FLINT Hologuess was faster on three of the four dense cases. Ore was faster on `Dense_r5d25`. The larger corpus starts from the [original D-algebraic Guessing data](https://github.com/T3gu1a/D-algebraic-Guessing/tree/19bc0b7e862d4af4b4a3ed713cf0511875fe62f0/benchmarks). [First differential race](../benchmarks/results/2026-11-26-flint-races/report.md#first-differential-loss-from-the-prior-cohort) · [Three additional dense cases](../benchmarks/results/2026-11-26-flint-dense-next3/report.md) · [Corpus report](../benchmarks/results/2026-11-26-upstream/report.md)
- **Constant-coefficient recurrences.** Hologuess was faster than native Ore on the Fibonacci and order-three inputs. SymPy and Mathematica's `FindLinearRecurrence` were faster on those same two inputs. The report records which other native engines could be run. [Constant-coefficient races](../benchmarks/results/2026-11-26-constant-recurrence-races/report.md)

The full source inventories identify upstream calls and the exact compatibility tests that exercise them. [Ore algebra and SymPy](../tests/upstream_ore_inventory.json) · [Maple gfun](../tests/upstream_gfun_inventory.json) · [Mathematica RISC](../tests/upstream_risc_inventory.json) · [FriCAS](../tests/upstream_fricas_inventory.json)